# Logistic Regression — Parameters, Metrics & Cross-Validation
## DA5401W - Data Analytics Lab
**Instructor:** Dr. Arun B Ayyar

---

## About This Tutorial

This tutorial guides you through the key parameters of scikit-learn's `LogisticRegression` class, classification evaluation metrics, and cross-validation techniques for building better models. Each problem has a detailed problem statement, pre-written data setup code, a solution cell with hints for you to complete, and a visualization cell that explains the results.

## Table of Contents

| Problem | Topic | Dataset |
|---------|-------|---------|
| **1** | Effect of `C` (Regularization Strength) | Breast Cancer |
| **2** | Comparing `solver` and `penalty` Parameters | Breast Cancer |
| **3** | Classification Metrics: Confusion Matrix, Precision, Recall, F1, AUC-ROC | Breast Cancer |
| **4** | Handling Imbalanced Classes with `class_weight` | Synthetic Imbalanced |
| **5** | Multi-class Classification with `multi_class` | Iris |
| **6** | Cross-Validation and Hyperparameter Tuning | Wine |

---

## Setup — Run This First

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.datasets import load_breast_cancer, load_iris, load_wine
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)
print('All libraries loaded successfully!')

---
## Problem 1: Effect of `C` (Regularization Strength)

### Problem Statement

In scikit-learn's `LogisticRegression`, the parameter `C` controls the **inverse of regularization strength**. This is the opposite of `alpha` in Ridge/Lasso:

> **Small C** → Strong regularization → Simpler model → May underfit  
> **Large C** → Weak regularization → Complex model → May overfit  

The relationship is: `C = 1 / alpha`, so `C=0.001` is equivalent to very strong regularization and `C=1000` is almost no regularization.

**What you need to do:**

Using the **Breast Cancer dataset** (30 features, binary classification), fit Logistic Regression models for each `C` value in `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`. For each model, record:
- **Train accuracy** and **Test accuracy**
- **Number of non-zero coefficients** (use `penalty='l1'` and `solver='liblinear'` to enable sparsity)

**Key scikit-learn parameters for LogisticRegression:**

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `C` | float | 1.0 | Inverse regularization strength. **Smaller = more regularization** |
| `penalty` | str | 'l2' | Type of regularization: 'l1', 'l2', 'elasticnet', 'None' |
| `solver` | str | 'lbfgs' | Optimization algorithm (must be compatible with penalty) |
| `max_iter` | int | 100 | Max iterations — **increase to 1000** to ensure convergence |
| `fit_intercept` | bool | True | Whether to add a bias term |

**Solver-Penalty compatibility:**

| Solver | L1 | L2 | ElasticNet | None |
|--------|----|----|------------|------|
| `lbfgs` | ✗ | ✓ | ✗ | ✓ |
| `liblinear` | ✓ | ✓ | ✗ | ✗ |
| `saga` | ✓ | ✓ | ✓ | ✓ |
| `newton-cg` | ✗ | ✓ | ✗ | ✓ |
| `sag` | ✗ | ✓ | ✗ | ✓ |

**Expected outcome:** As C increases, training accuracy increases but test accuracy peaks at an optimal C.

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Dataset: {cancer.target_names[0]} vs {cancer.target_names[1]}')
print(f'Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')
print(f'Number of features: {X.shape[1]}')
print(f'Class distribution (train): {dict(zip(*np.unique(y_train, return_counts=True)))}')

In [ ]:
# ── YOUR SOLUTION ──────────────────────────────────────────────────────────
# Hint: Loop over C_values, fit a LogisticRegression with penalty='l1',
#       solver='liblinear', max_iter=1000, and the current C value.
#       Record train accuracy, test accuracy, and number of non-zero coefficients.

C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

train_acc = []
test_acc  = []
nonzero   = []

for C in C_values:
    # Step 1: Create LogisticRegression with penalty='l1', solver='liblinear',
    #         max_iter=1000, and the current C value
    model = # YOUR CODE HERE

    # Step 2: Fit the model on scaled training data
    # YOUR CODE HERE

    # Step 3: Compute train and test accuracy
    tr_acc = # YOUR CODE HERE
    te_acc = # YOUR CODE HERE

    # Step 4: Count non-zero coefficients
    #         Hint: model.coef_ contains the coefficients; use np.sum(... != 0)
    nz = # YOUR CODE HERE

    train_acc.append(tr_acc)
    test_acc.append(te_acc)
    nonzero.append(nz)

# Print summary table
print(f"{'C':>8}  {'Train Acc':>10}  {'Test Acc':>10}  {'Non-zero Coefs':>16}")
print('-' * 52)
for c, tr, te, nz in zip(C_values, train_acc, test_acc, nonzero):
    print(f"{c:>8}  {tr:>10.4f}  {te:>10.4f}  {nz:>16}")

In [ ]:
# ── VISUALIZATION ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogx(C_values, train_acc, 'o-', lw=2.5, ms=8, color='#3498DB', label='Train Accuracy')
axes[0].semilogx(C_values, test_acc,  's-', lw=2.5, ms=8, color='#E74C3C', label='Test Accuracy')
axes[0].set_xlabel('C (log scale)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[0].set_title('Effect of C on Train vs Test Accuracy', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].semilogx(C_values, nonzero, 'D-', lw=2.5, ms=8, color='#2ECC71')
axes[1].set_xlabel('C (log scale)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Number of Non-zero Coefficients', fontsize=11, fontweight='bold')
axes[1].set_title('C vs Feature Sparsity (L1 penalty)', fontsize=12, fontweight='bold')
axes[1].axhline(X.shape[1], color='gray', ls='--', lw=1.5, label='All features')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Problem 1: C Parameter in Logistic Regression\n'
             'Small C = strong regularization; Large C = weak regularization',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Problem 2: Comparing `solver` and `penalty` Parameters

### Problem Statement

scikit-learn's `LogisticRegression` supports multiple **solvers** (optimization algorithms) and **penalty** types. Choosing the right combination affects:
- **Convergence speed** (how fast the model trains)
- **Accuracy** (some solvers are more precise)
- **Sparsity** (only L1 penalty produces zero coefficients)

**Solvers explained:**

| Solver | Algorithm | Best For |
|--------|-----------|----------|
| `lbfgs` | Limited-memory BFGS | Small/medium datasets, L2 penalty |
| `liblinear` | Coordinate descent | Small datasets, L1 or L2 |
| `saga` | Stochastic Average Gradient | Large datasets, all penalties |
| `sag` | Stochastic Average Gradient | Large datasets, L2 only |
| `newton-cg` | Newton's method | Medium datasets, L2 only |

**What you need to do:**

Using the Breast Cancer dataset, compare the following 5 combinations of solver and penalty. For each, record: test accuracy and number of non-zero coefficients.

| # | Solver | Penalty | C |
|---|--------|---------|---|
| 1 | `lbfgs` | `l2` | 1.0 |
| 2 | `liblinear` | `l1` | 1.0 |
| 3 | `liblinear` | `l2` | 1.0 |
| 4 | `saga` | `l1` | 1.0 |
| 5 | `saga` | `elasticnet` | 1.0 |

For `elasticnet`, you also need to set `l1_ratio=0.5` (equal mix of L1 and L2).

**Expected outcome:** L1 penalty produces sparse models; L2 keeps all features. All solvers should achieve similar accuracy on this well-behaved dataset.

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# Using the same Breast Cancer dataset from Problem 1
configs = [
    {'solver': 'lbfgs',      'penalty': 'l2',         'l1_ratio': None, 'label': 'lbfgs + L2'},
    {'solver': 'liblinear',  'penalty': 'l1',         'l1_ratio': None, 'label': 'liblinear + L1'},
    {'solver': 'liblinear',  'penalty': 'l2',         'l1_ratio': None, 'label': 'liblinear + L2'},
    {'solver': 'saga',       'penalty': 'l1',         'l1_ratio': None, 'label': 'saga + L1'},
    {'solver': 'saga',       'penalty': 'elasticnet', 'l1_ratio': 0.5,  'label': 'saga + ElasticNet'},
]
print('Configurations to test:')
for i, cfg in enumerate(configs, 1):
    print(f'  {i}. {cfg["label"]}')

In [ ]:
# ── YOUR SOLUTION ──────────────────────────────────────────────────────────
# Hint: Loop over configs. For each, create a LogisticRegression with the
#       specified solver, penalty, C=1.0, max_iter=1000.
#       For elasticnet, also pass l1_ratio=cfg['l1_ratio'].

results = []

for cfg in configs:
    # Step 1: Build the model
    #   - If cfg['l1_ratio'] is not None, include l1_ratio=cfg['l1_ratio']
    #   - Always use C=1.0, max_iter=1000
    if cfg['l1_ratio'] is not None:
        model = # YOUR CODE HERE (include l1_ratio)
    else:
        model = # YOUR CODE HERE

    # Step 2: Fit and predict
    # YOUR CODE HERE

    # Step 3: Compute accuracy and non-zero coefficients
    acc = # YOUR CODE HERE
    nz  = # YOUR CODE HERE

    results.append({'Config': cfg['label'], 'Test Accuracy': round(acc, 4),
                    'Non-zero Coefs': nz, 'Total Coefs': X.shape[1]})

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

In [ ]:
# ── VISUALIZATION ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = [r['Config'] for r in results]
accs   = [r['Test Accuracy'] for r in results]
nzs    = [r['Non-zero Coefs'] for r in results]
colors = ['#3498DB', '#E74C3C', '#2ECC71', '#9B59B6', '#F39C12']

bars = axes[0].barh(labels, accs, color=colors, alpha=0.85, edgecolor='white')
axes[0].set_xlim(0.9, 1.01)
axes[0].set_xlabel('Test Accuracy', fontsize=11, fontweight='bold')
axes[0].set_title('Test Accuracy by Solver + Penalty', fontsize=12, fontweight='bold')
for bar, acc in zip(bars, accs):
    axes[0].text(acc + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{acc:.4f}', va='center', fontsize=9)
axes[0].grid(True, alpha=0.3, axis='x')

bars2 = axes[1].barh(labels, nzs, color=colors, alpha=0.85, edgecolor='white')
axes[1].axvline(X.shape[1], color='gray', ls='--', lw=1.5, label='All 30 features')
axes[1].set_xlabel('Non-zero Coefficients', fontsize=11, fontweight='bold')
axes[1].set_title('Feature Sparsity by Solver + Penalty', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
for bar, nz in zip(bars2, nzs):
    axes[1].text(nz + 0.3, bar.get_y() + bar.get_height()/2,
                 str(nz), va='center', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='x')

plt.suptitle('Problem 2: Solver & Penalty Comparison\n'
             'L1 penalty produces sparse models; L2 keeps all features',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Problem 3: Classification Metrics — Confusion Matrix, Precision, Recall, F1, AUC-ROC

### Problem Statement

Accuracy alone is not enough to evaluate a classifier. Depending on the problem, you may care more about:
- **Precision:** Of all predicted positives, how many are truly positive? (Avoid false alarms)
- **Recall (Sensitivity):** Of all actual positives, how many did we catch? (Avoid missing cases)
- **F1-Score:** Harmonic mean of Precision and Recall (balanced measure)
- **AUC-ROC:** Area under the ROC curve (overall discriminative ability)

**Key formulas:**

| Metric | Formula | Best Value |
|--------|---------|------------|
| Accuracy | (TP + TN) / Total | 1.0 |
| Precision | TP / (TP + FP) | 1.0 |
| Recall | TP / (TP + FN) | 1.0 |
| F1-Score | 2 × (Precision × Recall) / (Precision + Recall) | 1.0 |
| AUC-ROC | Area under ROC curve | 1.0 |

**What you need to do:**

Using the Breast Cancer dataset, fit a Logistic Regression model (C=1.0, solver='lbfgs', penalty='l2', max_iter=1000). Then compute ALL of the above metrics and:
1. Print the full `classification_report`
2. Plot the Confusion Matrix
3. Plot the ROC Curve with AUC
4. Plot the Precision-Recall Curve

**Key scikit-learn functions:**

| Function | Returns | Usage |
|----------|---------|-------|
| `confusion_matrix(y_true, y_pred)` | 2×2 matrix | TP, FP, FN, TN counts |
| `classification_report(y_true, y_pred)` | String | All metrics per class |
| `roc_curve(y_true, y_proba)` | fpr, tpr, thresholds | For ROC plot |
| `roc_auc_score(y_true, y_proba)` | float | AUC value |
| `precision_recall_curve(y_true, y_proba)` | precision, recall, thresholds | For PR plot |

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
# Using the same Breast Cancer dataset
print('Target: 0 = malignant, 1 = benign')
print(f'Test set size: {X_test.shape[0]} samples')
print(f'Test class distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}')

In [ ]:
# ── YOUR SOLUTION ──────────────────────────────────────────────────────────
# Step 1: Fit a LogisticRegression model (C=1.0, solver='lbfgs', penalty='l2', max_iter=1000)
model = # YOUR CODE HERE
# YOUR CODE HERE (fit)

# Step 2: Get predictions (class labels) and predicted probabilities
y_pred  = # YOUR CODE HERE  (use .predict())
y_proba = # YOUR CODE HERE  (use .predict_proba()[:, 1] for probability of positive class)

# Step 3: Compute all metrics
acc  = # YOUR CODE HERE
prec = # YOUR CODE HERE  (use average='binary')
rec  = # YOUR CODE HERE  (use average='binary')
f1   = # YOUR CODE HERE  (use average='binary')
auc  = # YOUR CODE HERE

print('=== CLASSIFICATION METRICS ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-Score  : {f1:.4f}')
print(f'AUC-ROC   : {auc:.4f}')
print('\n=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
# ── VISUALIZATION ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=cancer.target_names)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, lw=2.5, color='#E74C3C', label=f'AUC = {auc:.4f}')
axes[1].plot([0,1],[0,1], 'k--', lw=1.5, label='Random classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#E74C3C')
axes[1].set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
axes[1].set_ylabel('True Positive Rate (Recall)', fontsize=11, fontweight='bold')
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# Precision-Recall Curve
precision_c, recall_c, _ = precision_recall_curve(y_test, y_proba)
axes[2].plot(recall_c, precision_c, lw=2.5, color='#3498DB')
axes[2].fill_between(recall_c, precision_c, alpha=0.1, color='#3498DB')
axes[2].set_xlabel('Recall', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Precision', fontsize=11, fontweight='bold')
axes[2].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Problem 3: Classification Metrics Visualized\n'
             'ROC curve shows overall performance; PR curve is better for imbalanced data',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Problem 4: Handling Imbalanced Classes with `class_weight`

### Problem Statement

In many real-world problems, one class is much rarer than the other (e.g., fraud detection, disease diagnosis). A model trained on imbalanced data tends to **predict the majority class almost always**, achieving high accuracy but completely failing to detect the minority class.

scikit-learn's `LogisticRegression` has a `class_weight` parameter to address this:

| `class_weight` value | Effect |
|---------------------|--------|
| `None` (default) | All samples have equal weight — biased toward majority class |
| `'balanced'` | Automatically weights classes inversely proportional to frequency |
| `{0: w0, 1: w1}` | Manually specify weights for each class |

**What you need to do:**

A synthetic imbalanced dataset has been created with 950 samples of class 0 and 50 of class 1 (95:5 ratio). Fit two Logistic Regression models:
1. `class_weight=None` (default — ignores imbalance)
2. `class_weight='balanced'` (corrects for imbalance)

Compare their **Accuracy, Precision, Recall, and F1-score**. Pay special attention to Recall for the minority class (class 1).

**Key insight:** For imbalanced data, **Recall and F1-score** are more informative than Accuracy.

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
from sklearn.datasets import make_classification

X_imb, y_imb = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    weights=[0.95, 0.05], random_state=42
)

X_imb_tr, X_imb_te, y_imb_tr, y_imb_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)
sc_imb = StandardScaler()
X_imb_tr_s = sc_imb.fit_transform(X_imb_tr)
X_imb_te_s = sc_imb.transform(X_imb_te)

unique, counts = np.unique(y_imb_tr, return_counts=True)
print('Imbalanced Dataset Created!')
print(f'Training class distribution: Class 0 = {counts[0]}, Class 1 = {counts[1]}')
print(f'Imbalance ratio: {counts[0]/counts[1]:.1f}:1')

In [ ]:
# ── YOUR SOLUTION ──────────────────────────────────────────────────────────
# Step 1: Fit model WITHOUT class_weight correction (class_weight=None)
model_default = # YOUR CODE HERE (solver='lbfgs', C=1.0, max_iter=1000)
# YOUR CODE HERE (fit)

# Step 2: Fit model WITH class_weight='balanced'
model_balanced = # YOUR CODE HERE (add class_weight='balanced')
# YOUR CODE HERE (fit)

# Step 3: Compute metrics for both models
# For each model, compute: accuracy, precision, recall, f1
# Use average='binary' for binary classification

for name, model in [('Default (no weight)', model_default),
                     ('Balanced class_weight', model_balanced)]:
    y_pred = # YOUR CODE HERE
    acc  = # YOUR CODE HERE
    prec = # YOUR CODE HERE
    rec  = # YOUR CODE HERE
    f1   = # YOUR CODE HERE
    print(f'\n--- {name} ---')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}  <- Most important for minority class!')
    print(f'  F1-Score  : {f1:.4f}')
    print(classification_report(y_imb_te, y_pred, target_names=['Majority (0)', 'Minority (1)']))

In [ ]:
# ── VISUALIZATION ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model), color in zip(
    axes,
    [('Default', model_default), ('Balanced', model_balanced)],
    ['#E74C3C', '#2ECC71']
):
    cm = confusion_matrix(y_imb_te, model.predict(X_imb_te_s))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0', 'Pred 1'],
                yticklabels=['True 0', 'True 1'])
    rec_minority = cm[1,1] / (cm[1,0] + cm[1,1]) if (cm[1,0]+cm[1,1]) > 0 else 0
    ax.set_title(f'{name}\nMinority Recall = {rec_minority:.2f}',
                 fontsize=12, fontweight='bold')

plt.suptitle('Problem 4: class_weight Effect on Imbalanced Data\n'
             'Balanced weighting dramatically improves minority class recall',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Problem 5: Multi-class Classification with `multi_class`

### Problem Statement

Logistic Regression is naturally a binary classifier. For multi-class problems, scikit-learn supports two strategies via the `multi_class` parameter:

| Strategy | How It Works | When to Use |
|----------|-------------|-------------|
| `'ovr'` (One-vs-Rest) | Trains K binary classifiers, one per class | Default for most solvers |
| `'multinomial'` | Trains a single model with softmax over all K classes | Better for correlated classes |
| `'auto'` | Chooses based on solver and data | General default |

**What you need to do:**

Using the **Iris dataset** (3 classes: setosa, versicolor, virginica), fit two models:
1. `solver='liblinear'`
2. `solver='lbfgs'`

For each model:
- Report per-class Precision, Recall, and F1-score using `classification_report`
- Plot the confusion matrix

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_iris_tr, X_iris_te, y_iris_tr, y_iris_te = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)
sc_iris = StandardScaler()
X_iris_tr_s = sc_iris.fit_transform(X_iris_tr)
X_iris_te_s = sc_iris.transform(X_iris_te)

print(f'Iris Dataset: {len(iris.target_names)} classes')
print(f'Classes: {list(iris.target_names)}')
print(f'Training samples: {X_iris_tr.shape[0]}, Test samples: {X_iris_te.shape[0]}')

In [ ]:
# ── YOUR SOLUTION ──────────────────────────────────────────────────────────
# Step 1: Fit OvR model
#   - solver='liblinear', solver='liblinear', C=1.0, max_iter=1000
model_ovr = # YOUR CODE HERE
# YOUR CODE HERE (fit)

# Step 2: Fit Multinomial model
#   - solver='lbfgs', solver='lbfgs', C=1.0, max_iter=1000
model_mn = # YOUR CODE HERE
# YOUR CODE HERE (fit)

# Step 3: Print classification report for both models
for name, model in [('OvR (One-vs-Rest)', model_ovr), ('Multinomial', model_mn)]:
    y_pred = # YOUR CODE HERE
    acc = # YOUR CODE HERE
    print(f'\n--- {name} --- (Accuracy: {acc:.4f})')
    print(classification_report(y_iris_te, y_pred, target_names=iris.target_names))

In [ ]:
# ── VISUALIZATION ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, [('OvR', model_ovr), ('Multinomial', model_mn)]):
    cm = confusion_matrix(y_iris_te, model.predict(X_iris_te_s))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=iris.target_names, yticklabels=iris.target_names)
    acc = accuracy_score(y_iris_te, model.predict(X_iris_te_s))
    ax.set_title(f'{name} — Accuracy: {acc:.4f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('True', fontsize=10)

plt.suptitle('Problem 5: Multi-class Classification\nOvR vs Multinomial Confusion Matrices',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Problem 6: Cross-Validation and Hyperparameter Tuning

### Problem Statement

Cross-validation is the correct way to evaluate and tune a model. Instead of a single train/test split, k-fold CV splits the training data into k folds, trains on k-1 folds, and validates on the remaining fold — repeating k times.

For classification, use **Stratified K-Fold** which preserves the class proportion in each fold.

**What you need to do:**

Using the **Wine dataset** (3 classes, 13 features):

1. Use `cross_val_score` with `StratifiedKFold(n_splits=5)` to evaluate a Logistic Regression model using 4 different scoring metrics: `'accuracy'`, `'precision_macro'`, `'recall_macro'`, `'f1_macro'`

2. Use `GridSearchCV` to find the best combination of `C` and `penalty` from:
   - `C`: [0.01, 0.1, 1, 10, 100]
   - `penalty`: ['l1', 'l2']
   - `solver`: 'saga', `max_iter`: 2000, `cv=5`, `scoring='f1_macro'`

3. Report the best parameters and the best CV score

**Key functions:**

| Function | Description |
|----------|-------------|
| `cross_val_score(model, X, y, cv=skf, scoring='accuracy')` | Returns array of CV scores |
| `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` | Stratified CV splitter |
| `GridSearchCV(model, param_grid, cv=5, scoring='f1_macro')` | Exhaustive hyperparameter search |
| `.best_params_` | Best parameters found by GridSearchCV |
| `.best_score_` | Best CV score found by GridSearchCV |

In [ ]:
# ── DATA CREATION ──────────────────────────────────────────────────────────
wine = load_wine()
X_wine, y_wine = wine.data, wine.target

X_wine_tr, X_wine_te, y_wine_tr, y_wine_te = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)
sc_wine = StandardScaler()
X_wine_tr_s = sc_wine.fit_transform(X_wine_tr)
X_wine_te_s = sc_wine.transform(X_wine_te)

print(f'Wine Dataset: {len(wine.target_names)} classes, {X_wine.shape[1]} features')
print(f'Classes: {list(wine.target_names)}')
print(f'Training samples: {X_wine_tr.shape[0]}')

In [ ]:
# ── YOUR SOLUTION ──────────────────────────────────────────────────────────
# Part A: Cross-validation with multiple metrics

# Step 1: Create a StratifiedKFold object with 5 splits, shuffle=True, random_state=42
skf = # YOUR CODE HERE

# Step 2: Create a base LogisticRegression model (solver='lbfgs', C=1.0, max_iter=1000)
base_model = # YOUR CODE HERE

# Step 3: Evaluate using cross_val_score for each metric
metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
print('=== CROSS-VALIDATION RESULTS (5-fold Stratified) ===')
for metric in metrics:
    scores = # YOUR CODE HERE (use cross_val_score with cv=skf, scoring=metric)
    print(f'{metric:20s}: {scores.mean():.4f} +/- {scores.std():.4f}')

# Part B: GridSearchCV
print('\n=== GRID SEARCH FOR BEST HYPERPARAMETERS ===')

# Step 4: Define the parameter grid
param_grid = {
    'C':       # YOUR CODE HERE (list of C values: [0.01, 0.1, 1, 10, 100]),
    'penalty': # YOUR CODE HERE (list: ['l1', 'l2'])
}

# Step 5: Create GridSearchCV with solver='saga', max_iter=2000, cv=5, scoring='f1_macro'
grid_search = GridSearchCV(
    LogisticRegression(solver='saga', max_iter=2000, random_state=42),
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

# Step 6: Fit GridSearchCV on training data
# YOUR CODE HERE

# Step 7: Report best parameters and best CV score
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV F1-macro: {grid_search.best_score_:.4f}')

# Step 8: Evaluate best model on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_wine_te_s)
print(f'\nTest Accuracy (best model): {accuracy_score(y_wine_te, y_pred_best):.4f}')
print(f'Test F1-macro (best model): {f1_score(y_wine_te, y_pred_best, average="macro"):.4f}')

In [ ]:
# ── VISUALIZATION ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot GridSearch heatmap
cv_results = pd.DataFrame(grid_search.cv_results_)
pivot = cv_results.pivot_table(
    index='param_penalty', columns='param_C', values='mean_test_score'
)
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('GridSearchCV: F1-macro by C and Penalty', fontsize=12, fontweight='bold')
axes[0].set_xlabel('C value', fontsize=11)
axes[0].set_ylabel('Penalty', fontsize=11)

# Confusion matrix of best model
cm = confusion_matrix(y_wine_te, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=wine.target_names, yticklabels=wine.target_names)
axes[1].set_title(f'Best Model Confusion Matrix\nC={grid_search.best_params_["C"]}, '
                  f'penalty={grid_search.best_params_["penalty"]}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted', fontsize=10)
axes[1].set_ylabel('True', fontsize=10)

plt.suptitle('Problem 6: Cross-Validation & Hyperparameter Tuning\n'
             'GridSearch finds the optimal C and penalty combination',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary: Key scikit-learn Parameters for Logistic Regression

### `sklearn.linear_model.LogisticRegression` — Parameter Reference

| Parameter | Default | What It Controls | Recommended Setting |
|-----------|---------|-----------------|---------------------|
| `C` | 1.0 | Inverse regularization strength (1/alpha) | Tune via GridSearchCV |
| `penalty` | `'l2'` | Regularization type: 'l1', 'l2', 'elasticnet', None | 'l2' general; 'l1' for sparsity |
| `solver` | `'lbfgs'` | Optimization algorithm | See compatibility table |
| `max_iter` | 100 | Max iterations | **Always set to 1000+** |
| `class_weight` | None | Sample weighting for imbalanced classes | `'balanced'` for imbalanced data |
| `multi_class` | `'auto'` | Multi-class strategy: 'ovr', 'multinomial' | 'multinomial' for correlated classes |
| `l1_ratio` | None | ElasticNet mixing (0=L2, 1=L1) | Required when penalty='elasticnet' |
| `fit_intercept` | True | Whether to fit a bias term | Keep True unless data is centered |
| `random_state` | None | Seed for reproducibility | Set for reproducible results |

### Classification Metrics Quick Reference

| Metric | When to Use | Sensitive To |
|--------|-------------|-------------|
| Accuracy | Balanced classes | Class imbalance |
| Precision | When false positives are costly | Threshold |
| Recall | When false negatives are costly | Threshold |
| F1-Score | Balance precision and recall | Class imbalance |
| AUC-ROC | Overall discriminative ability | Not threshold-dependent |

---
**Course:** DA5401W - Data Analytics Lab  |  **Instructor:** Dr. Arun B Ayyar